<div style="padding: 60px 40px; border-radius: 12px;  text-align: center;">

<h1 style="font-size: 2.4em; font-weight: 800; margin-bottom: 16px; letter-spacing: -0.5px;">LLM-for-LLM</h1>
<h2 style="font-size: 1.4em; font-weight: 400; color: #1a6bb5; margin-bottom: 32px;">Using Agentic AI to Optimize Critical Foundation Model Workloads</h2>
<h2 style="font-size: 1.2em; font-weight: 400; color: #1a1a1a; margin-bottom: 32px;">Vincent Fang · AMD AIG</h2>
<hr style="border-color: rgba(0,0,0,0.15); margin: 24px 0;">
<h5 style="color: #555555; font-size: 1.05em;">Platform: AMD ROCm &nbsp;&middot;&nbsp; Runtime: vLLM &nbsp;&middot;&nbsp; GPU: Radeon™ RX 7900 XTX · GFX1100 · Wuxi Radeon Cloud</p>
</div>

## Workshop Roadmap

Agentic AI is quickly becoming a practical tool for inference optimization. Strong models already carry useful knowledge of GPU kernels, runtimes, profilers, and tuning workflows. On ROCm and AMD GPUs, that knowledge can now be applied end to end on a real serving stack.

| # | Topic | Format |
|---|-------|--------|
| 1 | Why agentic optimization is becoming practical | Opening context |
| 2 | What the AMD ROCm + vLLM inference stack looks like | Architecture walkthrough |
| 3 | Approach: agentic optimization with `vllm-optimize` | Workflow overview |
| 4 | What the agent actually finds in profiler traces | Live data |
| 5 | How TunableOps performs offline GEMM kernel selection | Technical deep dive |
| 6 | End-to-end results on Qwen3-8B | Benchmark charts |
| **7** | **Live demo: the agent optimizes a model in real time** | **OpenCode** |


---
## 1 · Why Agentic Optimization Is Becoming Practical

### A concrete result

In this workshop, the agent improves Qwen3-8B throughput from **356 TPS to 565 TPS** at medium concurrency.

> **Result: +58.4% throughput, or ~37% lower cost per generated token.**

For a serving workload with **1M requests/day** and **1024 output tokens/request**:

| Scenario | Throughput gain | Cost per token | Approx. yearly savings* |
|----------|-----------------|----------------|--------------------------|
| Typical tuning win | +10% | ~9% lower | ~$2.6K |
| Strong tuning win | +50% | ~33% lower | ~$13K |
| **This workshop** | **+58%** | **~37% lower** | **~$15K** |

<small>*Illustrative estimate using a $3/hr GPU rate. The key point is the cost-per-token reduction, not the exact cloud price.</small>

### Why the workflow matters

The bottleneck spans vLLM batching, GEMM dispatch, ROCm libraries, and AMD GPU architecture. Getting a safe win requires evidence from the actual workload: benchmark results, profiler traces, real GEMM shapes, and end-to-end verification.

> This workshop shows how an LLM agent can run that loop on ROCm: **measure -> analyze -> tune -> verify**.


---
## 2 · vLLM + ROCm Inference Architecture

vLLM has native ROCm support, so developers can use familiar serving APIs and high-level inference features on AMD GPUs, from Instinct-class datacenter accelerators to Radeon developer workstations.

![vLLM + ROCm inference architecture](https://raw.githubusercontent.com/AMD-AIM/inference-skill/refs/heads/refactor-vllm/assetsvllm_rocm_architecture_fresh.png)

**Takeaway:** vLLM provides the serving abstraction; ROCm connects that abstraction to AMD software libraries and GPU hardware. The optimization work in this workshop happens inside that stack, using evidence from the real workload.


---
## 3 · Approach: The `vllm-optimize` Skill

`vllm-optimize` packages the performance workflow into a repeatable loop: it benchmarks the model, captures evidence, tunes the measured bottleneck, verifies the result, and leaves behind a report developers can inspect.

<div style="margin: 18px 0 8px; padding: 28px 30px; border-radius: 18px; background: linear-gradient(135deg, #f7fbff 0%, #eef6ff 100%); border: 1px solid #d8e8f7;">
  <div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 18px;">
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #5aa7d6; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #39779f; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">PHASE 0</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Environment</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">Detect GPU, ROCm, and vLLM version</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">~10s</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #62bf8e; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #3b8b64; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">PHASE 1</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Server Setup</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">Start vLLM with profiler and TunableOps recording</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">~2min</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #f4a64b; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #b66a14; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">PHASE 2</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Bench + Profile</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">Run concurrency sweep and collect traces</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">~12-15min</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #7c8ee6; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #5867c9; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">PHASE 3</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Analysis</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">Find GPU-time bottlenecks and optimization targets</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">~1min</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #b476bf; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #8b4d96; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">PHASE 4</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Kernel Optimize</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">TunedOps / Triton Optimize / GEAK Agent</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">~10min</div>
    </div>
    <div style="background: white; border: 1px solid #d7e5f2; border-top: 7px solid #e36b6b; border-radius: 16px; padding: 18px; min-height: 138px; box-shadow: 0 8px 20px rgba(25, 50, 90, 0.08);">
      <div style="color: #bf4747; font-weight: 800; font-size: 0.88em; letter-spacing: 0.04em;">PHASE 5-6</div>
      <div style="font-size: 1.18em; font-weight: 800; margin-top: 8px; color: #102033;">Verify + Report</div>
      <div style="color: #526173; margin-top: 8px; line-height: 1.35;">Inject tuned kernels, run E2E checks, summarize results</div>
      <div style="color: #7a8aa0; margin-top: 12px; font-size: 0.9em;">~8min + report</div>
    </div>
  </div>
</div>

**Outcome:** about **35 minutes** from baseline measurement to an optimized serving configuration, with artifacts and a final report for review.


---
## 4 · Find the Bottleneck

After the baseline run, the agent profiles the serving workload and asks one question first: **where is the GPU time going?**

For this Qwen3-8B run at `conc=16`, the decode phase is dominated by GEMM kernels. That makes GEMM the first optimization target, instead of guessing flags or tuning unrelated parts of the stack.

### GPU Kernel Breakdown — Qwen3-8B, conc=16, decode phase


![GPU kernel time breakdown](https://raw.githubusercontent.com/AMD-AIM/inference-skill/refs/heads/refactor-vllm/assetspart4_gpu_kernel_breakdown.png)


### Diagnosis

<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 14px 0 8px;">
  <div style="background:#fff5f5; border:1px solid #ffd7d7; border-radius:14px; padding:18px;">
    <div style="font-size:1.6em; font-weight:800; color:#c53030;">91%</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">GPU active time</div>
    <div style="color:#5b6678; margin-top:6px;">spent in GEMM kernels</div>
  </div>
  <div style="background:#f5f9ff; border:1px solid #d7e7ff; border-radius:14px; padding:18px;">
    <div style="font-size:1.6em; font-weight:800; color:#2b6cb0;">decode</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">phase bottleneck</div>
    <div style="color:#5b6678; margin-top:6px;">visible at medium concurrency</div>
  </div>
  <div style="background:#f7fff8; border:1px solid #cbe8d0; border-radius:14px; padding:18px;">
    <div style="font-size:1.6em; font-weight:800; color:#2f855a;">GEMM</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">optimization target</div>
    <div style="color:#5b6678; margin-top:6px;">focus the next phase here</div>
  </div>
</div>

The important takeaway is the workflow: the agent uses profiler evidence to narrow the problem before applying any optimization.


---
## 5 · Fix the Bottleneck

Once GEMM is identified as the dominant cost, the agent moves into a kernel-level optimization path. In this example, the first step is **offline GEMM kernel selection with TunableOps**.

<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 16px 0 18px;">
  <div style="background:#ffffff; border:1px solid #dbe7f3; border-top:6px solid #5aa7d6; border-radius:14px; padding:18px;">
    <div style="font-weight:800; color:#1d4f73;">1 · Record</div>
    <div style="color:#566274; margin-top:8px; line-height:1.35;">Capture GEMM calls observed during real serving traffic.</div>
  </div>
  <div style="background:#ffffff; border:1px solid #dbe7f3; border-top:6px solid #b476bf; border-radius:14px; padding:18px;">
    <div style="font-weight:800; color:#743a80;">2 · Tune</div>
    <div style="color:#566274; margin-top:8px; line-height:1.35;">Benchmark candidate rocBLAS / hipBLASLt kernels offline.</div>
  </div>
  <div style="background:#ffffff; border:1px solid #dbe7f3; border-top:6px solid #62bf8e; border-radius:14px; padding:18px;">
    <div style="font-weight:800; color:#2f6f4e;">3 · Inject</div>
    <div style="color:#566274; margin-top:8px; line-height:1.35;">Load the selected kernels at serving time without changing vLLM source.</div>
  </div>
</div>

Beyond TunableOps, the same workflow can extend to a **Triton Kernel Agent** for more complex operator fusion, and can optionally enable **GEAK** for more advanced HIP kernel generation. Across these paths, the design principle is the same: keep the optimization pluggable and scoped to the measured bottleneck, with **no changes to vLLM source or model structure**.

### Kernel-Level Result


![Kernel optimization result](https://raw.githubusercontent.com/AMD-AIM/inference-skill/refs/heads/refactor-vllm/assetspart5_kernel_optimize_result.png)


---
## 6 · Measure the Gain

The final check is end-to-end serving throughput. A microbenchmark improvement only matters if it survives integration and improves the real vLLM workload.

<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; margin: 14px 0 18px;">
  <div style="background:#f6f8ff; border:1px solid #dce5ff; border-radius:14px; padding:18px;">
    <div style="font-size:1.55em; font-weight:800; color:#2b6cb0;">356 TPS</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">baseline</div>
    <div style="color:#5b6678; margin-top:6px;">conc=16</div>
  </div>
  <div style="background:#f4fff7; border:1px solid #ccebd6; border-radius:14px; padding:18px;">
    <div style="font-size:1.55em; font-weight:800; color:#2f855a;">565 TPS</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">optimized</div>
    <div style="color:#5b6678; margin-top:6px;">same workload</div>
  </div>
  <div style="background:#fff8ef; border:1px solid #f7d7ad; border-radius:14px; padding:18px;">
    <div style="font-size:1.55em; font-weight:800; color:#c05621;">+58.4%</div>
    <div style="font-weight:700; color:#202938; margin-top:4px;">throughput</div>
    <div style="color:#5b6678; margin-top:6px;">~37% lower cost/token</div>
  </div>
</div>

### Benchmark configuration

| Item | Setting |
|------|---------|
| Model | Qwen3-8B (`hidden=4096`, `layers=36`, `heads=32`, `intermediate=12288`) |
| GPU | AMD RDNA3 gfx1100 · 48 WGP / 96 CU · GDDR6 |
| vLLM | `--enforce-eager`, TP=1, `max_model_len=4096` |
| dtype | bfloat16 |
| Input / output length | ISL=1024, OSL=1024 tokens |


![End-to-end throughput result](https://raw.githubusercontent.com/AMD-AIM/inference-skill/refs/heads/refactor-vllm/assetspart6_e2e_throughput.png)


---
## 7 · Live Demo

<div style=" padding: 32px; border-radius: 10px; ">

<h3 style="color: #1a6bb5; margin-top: 0;">Hands-On: Try the Workflow Yourself</h3>

<div style="background: rgba(0,0,0,0.03); border: 1px solid rgba(0,100,200,0.2); border-radius: 12px; padding: 18px 22px; margin: 18px 0 22px;">
  <div style="display: grid; grid-template-columns: 160px 1fr; row-gap: 12px; column-gap: 18px; align-items: baseline;">
    <div style="color: #1a6bb5; font-weight: 700;">Environment</div>
    <div style="color: #1a1a1a;">Wuxi Radeon Pro cluster</div>
    <div style="color: #1a6bb5; font-weight: 700;">Capacity</div>
    <div style="color: #1a1a1a;">160+ concurrent users</div>
    <div style="color: #1a6bb5; font-weight: 700;">Access URL</div>
    <div style="color: #1a5faa; font-weight: 700; word-break: break-all;">https://radeon.oneclickamd.ai/</div>
  </div>
</div>

<p style="color: #333333;">The goal of this last section is not just to watch a demo. It is for everyone to launch a ROCm-ready instance, run the workflow end to end, and inspect the evidence at each phase boundary.</p>

<h4 style="color: #1a6bb5;">Suggested flow:</h4>

<ol style="color: #1a1a1a; line-height: 2;">
<li>Open <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">https://radeon.oneclickamd.ai/</code> and select the <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">Agentic Inference Optimization</code> from gallery</li></li> 
<li>Open OpenCode in terminal: <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">opencode</code></li>
<li>Select an agent model, for example <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">/models Claude Sonnet 4.6</code></li>
<li>Invoke the skill: <code style="background:#deeaf7; padding:2px 6px; border-radius:4px;">like /vllm-optimize qwen/Qwen3-8B or /vllm-optimize qwen/Qwen3.5-4B</code></li>
<li>Enable demo mode and review the evidence as each phase completes</li>
<li>Once you are comfortable with the flow, repeat it on your own model or config</li>
</ol>

</div>